# 09 — Esperimenti anti-overfitting

Notebook Colab per lanciare la matrice di prove sul branch
`experiments/overfitting-reduction` **senza sovrascrivere** i run precedenti.

Ogni prova usa `--run-name`, quindi checkpoint e metriche 3D finiscono in
cartelle dedicate (`checkpoints/<run>/<arch>/`, `evaluation_outputs/<run>/`).
Alla fine si raccolgono best val fg-Dice, gap train-val e test fg-Dice in una
tabella di confronto salvata su Drive con timestamp (non sovrascrive).

**Ordine:** esegui le celle dall'alto verso il basso. Serve una GPU (Runtime -> Cambia tipo di runtime -> GPU).


## 1. Monta Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Clona il repo (branch degli esperimenti) e installa le dipendenze


In [ ]:
import os

REPO_URL = 'https://github.com/Drastid/BraTS-PEDs-brain-tumour-segmentation.git'
BRANCH   = 'experiments/overfitting-reduction'
REPO_DIR = '/content/BraTS-PEDs-brain-tumour-segmentation'

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!git fetch --all -q && git checkout $BRANCH && git pull -q origin $BRANCH
!git log --oneline -1
!pip install -q -r requirements.txt


## 3. Porta il dataset preprocessato su disco locale (NVMe)

Legge lo zip da Drive **una volta** e lo scompatta in `/content/` (mai leggere
le migliaia di `.npy` direttamente da Drive: affama la GPU). Salta se gia' presente.


In [ ]:
DATA_ROOT   = '/content/processed_dataset'
DATASET_ZIP = '/content/drive/MyDrive/BraTS_Project/processed_dataset.zip'  # <-- adegua se serve

if not os.path.isdir(DATA_ROOT):
    assert os.path.isfile(DATASET_ZIP), f'Zip non trovato: {DATASET_ZIP}'
    !cp "$DATASET_ZIP" /content/
    !unzip -q /content/processed_dataset.zip -d /content/
assert os.path.isdir(os.path.join(DATA_ROOT, 'train', 'images')), 'processed_dataset/train/images mancante'
print('Dataset pronto in', DATA_ROOT)


## 4. Configurazione degli esperimenti

`--run-name` isola gli output. `COMMON` (early stopping) e' applicato a tutte le
prove; poi ogni prova cambia **una leva alla volta** cosi' l'effetto e'
attribuibile. Itera prima su **U-Net** (veloce), poi porta la config vincente su
SegFormer/FPN aggiungendoli a `models`.

La metrica che conta e' se il **best val fg-Dice sale** rispetto al baseline
(guadagno di generalizzazione), non solo se il gap si stringe.


In [ ]:
CKPT_ROOT  = '/content/checkpoints'
BACKUP_DIR = '/content/drive/MyDrive/BraTS_Project/experiments'  # backup su Drive, cartella dedicata
APPLY_A100 = True   # riproduce la config del baseline (resnet50 / mit-b2 / crop224 / bf16)
FORCE_RERUN = False # True per riallenare anche i run gia' presenti (sovrascrive SOLO quel run-name)

# Leva comune a tutte le prove: taglia le epoche inutili senza rischi.
COMMON = ['--early-stopping', '--es-patience', '5']

# Matrice: una leva per riga. models=['unet'] per iterare veloce.
RUN_MATRIX = [
    {'run_name': 'exp1_earlystop', 'models': ['unet'], 'flags': []},
    {'run_name': 'exp2_wd',        'models': ['unet'], 'flags': ['--weight-decay', '5e-4']},
    {'run_name': 'exp3_enclr',     'models': ['unet'], 'flags': ['--encoder-lr-mult', '0.05']},
    {'run_name': 'exp4_aug',       'models': ['unet'], 'flags': ['--augment-prob', '0.6', '--augment-strength', '1.5']},
    # exp5_combo: compilalo dopo aver visto i risultati, es.:
    # {'run_name': 'exp5_combo', 'models': ['unet', 'segformer', 'fpn'],
    #  'flags': ['--encoder-lr-mult', '0.05', '--augment-prob', '0.6', '--augment-strength', '1.5']},
]
print(f'{len(RUN_MATRIX)} prove in coda')


## 5. Esegui la matrice

Guardia anti-sovrascrittura: se la cartella `checkpoints/<run>/` esiste gia',
la prova viene **saltata** (a meno di `FORCE_RERUN=True`). Cosi' puoi rilanciare
il notebook senza perdere i run fatti.


In [ ]:
import subprocess, sys, time

def run_experiment(run):
    run_name, models, flags = run['run_name'], run['models'], run['flags']
    out_dir = os.path.join(CKPT_ROOT, run_name)
    if os.path.isdir(out_dir) and not FORCE_RERUN:
        print(f'[skip] {run_name}: gia\' presente ({out_dir}). FORCE_RERUN=True per rifarlo.')
        return
    cmd = ['python', 'run_pipeline.py',
           '--data-root', DATA_ROOT, '--ckpt-root', CKPT_ROOT,
           '--run-name', run_name, '--models', *models,
           '--evaluate', '--backup-dir', BACKUP_DIR, *COMMON, *flags]
    if APPLY_A100:
        cmd.insert(2, '--apply-a100-config')
    print('\n' + '=' * 80)
    print(f'[run] {run_name}  models={models}  flags={flags}')
    print('  $ ' + ' '.join(cmd))
    print('=' * 80)
    t0 = time.time()
    # stream dell'output in tempo reale
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode != 0:
        print(f'[ERRORE] {run_name} uscito con codice {proc.returncode}')
    else:
        print(f'[ok] {run_name} in {(time.time()-t0)/60:.1f} min')

for run in RUN_MATRIX:
    run_experiment(run)


## 6. Raccogli i risultati in una tabella di confronto

Per ogni (run, modello): best val fg-Dice e in quale epoca, il gap train-val a
quell'epoca (quanto e' overfittato il checkpoint che *deployi*), e il test fg-Dice
3D. Se trova il baseline non isolato (`checkpoints/<arch>/`) lo aggiunge come
riferimento.


In [ ]:
import json, glob
import numpy as np
import pandas as pd

EVAL_ROOT = os.path.join(REPO_DIR, 'evaluation_outputs')

def read_history_best(hist_path):
    """Best val fg-Dice + epoca + gap train-val a quell'epoca."""
    if not os.path.isfile(hist_path):
        return None
    rows = json.load(open(hist_path))
    best = max(rows, key=lambda r: r.get('val_fg_dice', -1))
    val = best.get('val_fg_dice'); tr = best.get('train_fg_dice')
    gap = (tr - val) if (tr is not None and val is not None) else None
    return {'best_val_fg': val, 'best_epoch': best.get('epoch'),
            'train_fg_at_best': tr, 'gap_train_val': gap, 'n_epochs': len(rows)}

def read_test_fg(run_name, arch):
    """Test fg-Dice 3D dal json di evaluate_3d_test (cerca in run e default)."""
    cands = [os.path.join(EVAL_ROOT, run_name, f'test_3d_metrics_{arch}.json'),
             os.path.join(EVAL_ROOT, f'test_3d_metrics_{arch}.json')]
    for p in cands:
        if os.path.isfile(p):
            return json.load(open(p)).get('summary', {}).get('mean_fg_dice')
    return None

def collect(run_name, arch, label=None):
    hist = os.path.join(CKPT_ROOT, run_name, arch, 'history.json') if run_name \
           else os.path.join(CKPT_ROOT, arch, 'history.json')
    h = read_history_best(hist)
    if h is None:
        return None
    row = {'run': label or run_name, 'model': arch, **h,
           'test_fg': read_test_fg(run_name or '', arch)}
    return row

records = []
# baseline di riferimento, se presente (non isolato)
for arch in ['unet', 'fpn', 'segformer']:
    r = collect('', arch, label='baseline')
    if r: records.append(r)
# gli esperimenti
for run in RUN_MATRIX:
    for arch in run['models']:
        r = collect(run['run_name'], arch)
        if r: records.append(r)

df = pd.DataFrame(records)
if len(df):
    for c in ['best_val_fg', 'train_fg_at_best', 'gap_train_val', 'test_fg']:
        if c in df: df[c] = df[c].astype(float).round(4)
    df = df[['run','model','best_val_fg','best_epoch','train_fg_at_best',
             'gap_train_val','test_fg','n_epochs']].sort_values(['model','run'])
df


## 7. Salva la tabella su Drive (con timestamp, non sovrascrive)


In [ ]:
from datetime import datetime
os.makedirs(BACKUP_DIR, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_path = os.path.join(BACKUP_DIR, f'comparison_{stamp}.csv')
df.to_csv(csv_path, index=False)
print('Salvato ->', csv_path)


## 8. (Opzionale) Grafico: best val vs test per prova

Barre affiancate val/test: se una leva **alza la barra val** rispetto al
baseline, il gap era nocivo e stai guadagnando generalizzazione.


In [ ]:
import matplotlib.pyplot as plt

for arch in df['model'].unique():
    sub = df[df['model'] == arch]
    x = np.arange(len(sub)); w = 0.38
    fig, ax = plt.subplots(figsize=(1.6*len(sub)+3, 4))
    ax.bar(x - w/2, sub['best_val_fg'], w, label='best val fg-Dice')
    ax.bar(x + w/2, sub['test_fg'],     w, label='test fg-Dice')
    ax.set_xticks(x); ax.set_xticklabels(sub['run'], rotation=20, ha='right')
    ax.set_ylabel('fg-Dice'); ax.set_title(f'{arch} — confronto prove')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()
